# 📚 Building a Scientific-Paper Dataset — a Tutorial

This notebook teaches you how to build a clean, **legal** dataset of scientific papers and load it into a pandas `DataFrame` with these columns:

| column | what it is |
|---|---|
| `authors`  | list of author names |
| `title`    | paper title |
| `content`  | the **full text**, extracted from the PDF |
| `date`     | publication date |
| `country`  | country of the first author's institution |
| `journal`  | the journal / venue |
| `theme`    | a research-theme tag |

**No API key is required.** Everything here runs for free.

### What you'll learn
1. Where the papers come from and why this is legal
2. How to preview metadata before downloading anything
3. How to build the full dataset (metadata + full text + theme)
4. How to inspect, customize, and save your corpus
5. How to scale up, troubleshoot, and (optionally) upgrade the theme with an LLM

---
## 0. Background: where the data comes from

Instead of scraping a piracy site, we use **[OpenAlex](https://openalex.org)** — a free, open index of ~250 million scholarly works. For each paper it gives us structured metadata (authors, institutions, journal, date, topic) **and** a link to the *legal* open-access copy when one exists. We only ever download from those open-access links, so the corpus is clean by construction.

The whole process is a small pipeline, and all the code lives in **`pipeline.py`**:

```
OpenAlex API ──► metadata (authors, title, date, journal, country, theme)
      │           keep only papers with a legal open-access PDF
      ▼
Download PDF ──► from the open-access location (repositories preferred)
      ▼
PyMuPDF ──────► extract the text  →  the `content` column
      ▼
pandas ───────► one row per paper, then save to disk
```

**The `theme` column:** OpenAlex already classifies every paper into a *topic* (e.g. *"Single-cell and spatial transcriptomics"*). We use that as the theme for free. Section 7 shows how to swap in an LLM-generated tag if you ever want one.

## 1. Setup

Make sure the notebook is using the project's virtual environment as its **kernel** (top-right kernel picker → choose the one at `.venv`). The packages (`pandas`, `requests`, `pymupdf`) are already installed there.

The cell below imports the three functions we'll use from `pipeline.py`:
- `fetch_metadata` — get paper metadata only (fast, no downloads)
- `build_corpus`   — the full pipeline → a DataFrame
- `save_corpus`    — write the DataFrame to `.parquet` + `.csv`

In [1]:
import importlib, pipeline
importlib.reload(pipeline)  # re-run this cell after editing pipeline.py to pick up changes
from pipeline import fetch_metadata, build_corpus, save_corpus
import pandas as pd
print('ready ✅')

/Users/sam/Projects/sci_paper_llm/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


ready ✅


## 2. Step one: peek at the metadata (no downloads)

Before downloading dozens of PDFs, it's smart to check *what* your query returns. `fetch_metadata(n)` yields up to `n` papers as lightweight `Paper` objects — just the metadata, no file downloads. This is fast (one API call per 200 papers).

Run it and look at the table: are these the kinds of papers you want?

In [2]:
preview = list(fetch_metadata(20))   # 20 open-access papers, all fields

pd.DataFrame([
    {'title': p.title, 'theme': p.theme, 'country': p.country,
     'journal': p.journal, 'date': p.date}
    for p in preview
])

,title,theme,country,journal,date
0,Radiation Resistant Camera System for Monitori...,Magnetic confinement fusion research,JP,Plasma and Fusion Research,2020-06-08
1,PROTEIN MEASUREMENT WITH THE FOLIN PHENOL REAGENT,Glycosylation and Glycoproteins Research,US,Journal of Biological Chemistry,1951-11-01
2,Using thematic analysis in psychology,Community Health and Development,NZ,Qualitative Research in Psychology,2006-01-01
3,The PRISMA 2020 statement: an updated guidelin...,Meta-analysis and systematic reviews,AU,BMJ,2021-03-29
4,A short history of<i>SHELX</i>,Enzyme Structure and Function,DE,Acta Crystallographica Section A Foundations o...,2007-12-21
5,Fitting Linear Mixed-Effects Models Using <b>l...,Data Analysis with R,US,Journal of Statistical Software,2015-01-01
6,Preferred reporting items for systematic revie...,Meta-analysis and systematic reviews,CA,BMJ,2009-07-21
7,MizAR 60 for Mizar 50,Natural Language Processing Techniques,CZ,DROPS (Schloss Dagstuhl – Leibniz Center for I...,2023-01-01
8,ImageNet classification with deep convolutiona...,Advanced Neural Network Applications,CA,Communications of the ACM,2017-05-24
9,Gapped BLAST and PSI-BLAST: a new generation o...,Genomics and Phylogenetic Studies,US,Nucleic Acids Research,1997-09-01


Each `Paper` also carries fields we didn't show: `authors`, `abstract`, `doi`, `pdf_url`, and `pdf_candidates` (the ranked list of open-access PDF links the downloader will try). Inspect one fully:

In [3]:
p = preview[0]
print('title   :', p.title)
print('authors :', p.authors)
print('journal :', p.journal)
print('date    :', p.date)
print('country :', p.country, '  (all:', p.countries, ')')
print('theme   :', p.theme)
print('pdf urls:', p.pdf_candidates)

title   : Radiation Resistant Camera System for Monitoring Deuterium Plasma Discharges in the Large Helical Device
authors : ['M. Shoji', 'LHD Experiment Group']
journal : Plasma and Fusion Research
date    : 2020-06-08
country : JP   (all: ['JP'] )
theme   : Magnetic confinement fusion research
pdf urls: ['https://www.jstage.jst.go.jp/article/pfr/15/0/15_2402039/_pdf']


## 3. Step two: build the full dataset

`build_corpus` runs the whole pipeline and returns a DataFrame. The important arguments:

| argument | meaning |
|---|---|
| `n` | how many papers to fetch |
| `search` | free-text query (searches title/abstract/full text). `None` = every field |
| `extra_filters` | an OpenAlex filter string — see Section 5 for the cookbook |
| `with_fulltext` | `True` = download PDFs and extract `content` |
| `with_theme` | `False` = use OpenAlex's topic as the theme (**no API key**) |

We start with `n=25` and papers from 2022 onward (recent papers are more likely to have a working open-access PDF). Downloads are sequential, so this takes a minute or two — watch the log lines appear.

In [4]:
df = build_corpus(
    n=25,
    search=None,                                      # try 'machine learning' to focus a topic
    extra_filters='from_publication_date:2022-01-01',  # recent => better full-text coverage
    with_fulltext=True,
    with_theme=False,                                 # keyless theme from OpenAlex
)
df[['title', 'authors', 'country', 'journal', 'date', 'theme']]

2026-06-03 21:33:48,822 INFO [1/25] MizAR 60 for Mizar 50 | theme=Natural Language Processing Techniques | 78180 chars
2026-06-03 21:33:52,648 WARNING Not a PDF (text/html; charset=utf-8): https://open.icm.edu.pl/bitstreams/895ce6d1-be39-4129-bf0a-4fcc16ee2a18/download
2026-06-03 21:33:57,906 INFO [2/25] Two new Later Stone Age sites from the Final Pleistocene in  | theme=Diet and metabolism studies | 77101 chars
2026-06-03 21:33:57,948 WARNING Download/parse failed for https://onlinelibrary.wiley.com/doi/pdfdirect/10.3322/caac.21834: 403 Client Error: Forbidden for url: https://onlinelibrary.wiley.com/doi/pdfdirect/10.3322/caac.21834
2026-06-03 21:33:57,949 INFO [3/25] Global cancer statistics 2022: GLOBOCAN estimates of inciden | theme=Global Cancer Incidence and Screening | 0 chars
2026-06-03 21:33:57,983 WARNING Download/parse failed for https://onlinelibrary.wiley.com/doi/pdfdirect/10.3322/caac.21763: 403 Client Error: Forbidden for url: https://onlinelibrary.wiley.com/doi/pdfdire

,title,authors,country,journal,date,theme
0,MizAR 60 for Mizar 50,"[Jakubův, Jan, Chvalovský, Karel, Goertzel, Za...",CZ,DROPS (Schloss Dagstuhl – Leibniz Center for I...,2023-01-01,Natural Language Processing Techniques
1,Two new Later Stone Age sites from the Final P...,"[Ndiaye, Matar, Lespez, Laurent, Tribolo, Chan...",FR,ENLIGHTEN (Jurnal Bimbingan dan Konseling Islam),2024-03-28,Diet and metabolism studies
2,Global cancer statistics 2022: GLOBOCAN estima...,"[Freddie Bray, Mathieu Laversanne, Hyuna Sung,...",FR,CA A Cancer Journal for Clinicians,2024-04-04,Global Cancer Incidence and Screening
3,"Cancer statistics, 2023","[Rebecca L. Siegel, Kimberly D. Miller, Nikita...",US,CA A Cancer Journal for Clinicians,2023-01-01,Global Cancer Incidence and Screening
4,Global burden of bacterial antimicrobial resis...,"[Christopher J L Murray, Kevin S Ikuta, Fablin...",None,The Lancet,2022-01-19,Antibiotic Use and Resistance
5,A Multi-Modal Distributed Real-Time IoT System...,"[Khanam, Zeba, Achari, Vejey Pradeep Suresh, B...",GB,DROPS (Schloss Dagstuhl – Leibniz Center for I...,2024-01-01,Advanced Image and Video Retrieval Techniques
6,Aion Framework: Dimensional Emergence of AI Co...,"[T. B. Brown, Gao, Song]",US,DROPS (Schloss Dagstuhl – Leibniz Center for I...,2023-01-01,Topic Modeling
7,Accurate structure prediction of biomolecular ...,"[Josh Abramson, Jonas Adler, Jack Dunger, Rich...",GB,Nature,2024-05-08,Protein Structure and Dynamics
8,Suspending OpenMP Tasks on Asynchronous Events...,"[Romain Pereira, M. N. Martin, Adrien Roussel,...",FR,Lecture notes in computer science,2023-01-01,Parallel Computing and Optimization Techniques
9,The Coding Manual for Qualitative Researchers,[Maria Lungu],US,American Journal of Qualitative Research,2022-05-13,Evaluation and Performance Assessment


### Reading the log
Each line looks like `[7/25] Some Title... | theme=... | 41523 chars`. The char count is how much full text was extracted. A `WARNING` means a PDF couldn't be fetched (usually a publisher that blocks bots) — that paper simply gets an empty `content`. Expect roughly **75–90%** of papers to yield full text; that's normal and not an error.

## 4. Inspect your corpus

A few quick health checks: how many papers, how many have full text, and what themes showed up.

In [5]:
print('rows           :', len(df))
print('with full text :', df['content'].notna().sum())
print('unique themes  :', df['theme'].nunique())
print('\nmost common themes:')
print(df['theme'].value_counts().head(10))

rows           : 25
with full text : 16
unique themes  : 21

most common themes:
theme
Bioinformatics and Genomic Networks                    3
Global Cancer Incidence and Screening                  3
Climate change impacts on agriculture                  1
Diabetes, Cardiovascular Risks, and Lipoproteins       1
COVID-19 and healthcare impacts                        1
Gamma-ray bursts and supernovae                        1
Single-cell and spatial transcriptomics                1
Artificial Intelligence in Healthcare and Education    1
Qualitative Research Methods and Ethics                1
Alzheimer's disease research and treatments            1
Name: count, dtype: int64


Now actually read a paper. This grabs the first row that has full text and prints the start of it:

In [6]:
row = df[df['content'].notna()].iloc[0]
print(row['title'], '\u2014', row['theme'])
print('=' * 70)
print(row['content'][:1500])

MizAR 60 for Mizar 50 — Natural Language Processing Techniques
MizAR 60 for Mizar 50
Jan Jakubův #
Czech Technical University in Prague,
Czech Republic
Karel Chvalovský #
Czech Technical University in Prague,
Czech Republic
Zarathustra Goertzel
Czech Technical University in Prague,
Czech Republic
Cezary Kaliszyk #
Universität Innsbruck, Austria
INDRC, Prague, Czech Republic
Mirek Olšák
Institut des Hautes Études Scientifiques,
Paris, France
Bartosz Piotrowski #
Czech Technical University in Prague,
Czech Republic
Stephan Schulz
DHBW Stuttgart, Germany
Martin Suda #
Czech Technical University in Prague,
Czech Republic
Josef Urban
Czech Technical University in Prague,
Czech Republic
Abstract
As a present to Mizar on its 50th anniversary, we develop an AI/TP system that automatically
proves about 60 % of the Mizar theorems in the hammer setting. We also automatically prove 75 %
of the Mizar theorems when the automated provers are helped by using only the premises used in the
human-written

## 5. Customize your corpus — a filter cookbook

Shape the dataset with `search=` (free text) and `extra_filters=` (OpenAlex filters). Combine multiple filters by separating them with commas in one string. Common recipes:

```python
# A specific topic
build_corpus(50, search='CRISPR gene editing')

# A date window
build_corpus(50, extra_filters='from_publication_date:2023-01-01,to_publication_date:2023-12-31')

# Only English-language articles
build_corpus(50, extra_filters='from_publication_date:2022-01-01,language:en')

# Papers from a specific country's institutions (ISO code, lowercase)
build_corpus(50, extra_filters='authorships.institutions.country_code:us')

# Only journal articles (exclude preprints, datasets, etc.)
build_corpus(50, extra_filters='type:article')
```

Edit the cell below and run it to try your own combination. (Browse all filter fields in the [OpenAlex docs](https://docs.openalex.org/api-entities/works/filter-works).)

In [7]:
my_df = build_corpus(
    n=15,
    search='climate change',
    extra_filters='from_publication_date:2023-01-01,type:article',
    with_fulltext=True,
    with_theme=False,
)
my_df[['title', 'country', 'journal', 'date', 'theme']]

2026-06-03 21:42:11,923 WARNING Not a PDF (text/html; charset=utf-8): https://www.econstor.eu/bitstream/10419/288045/1/JOFI_JOFI13219.pdf
2026-06-03 21:42:11,981 WARNING Download/parse failed for https://onlinelibrary.wiley.com/doi/pdfdirect/10.1111/jofi.13219: 403 Client Error: Forbidden for url: https://onlinelibrary.wiley.com/doi/pdfdirect/10.1111/jofi.13219
2026-06-03 21:42:11,982 INFO [1/15] Firm‐Level Climate Change Exposure | theme=Market Dynamics and Volatility | 0 chars
2026-06-03 21:42:17,601 INFO [2/15] IPCC, 2023: Climate Change 2023: Synthesis Report, Summary f | theme=COVID-19 impact on air quality | 147782 chars
2026-06-03 21:42:24,512 INFO [3/15] A global transition to flash droughts under climate change | theme=Hydrology and Drought Analysis | 37972 chars
2026-06-03 21:42:27,161 INFO [4/15] Climate Change, Firm Performance, and Investor Surprises | theme=Market Dynamics and Volatility | 199994 chars
2026-06-03 21:42:27,689 WARNING Not a PDF (text/html; charset=utf-8): 

,title,country,journal,date,theme
0,Firm‐Level Climate Change Exposure,CN,The Journal of Finance,2023-02-28,Market Dynamics and Volatility
1,"IPCC, 2023: Climate Change 2023: Synthesis Rep...",None,None,2023-07-21,COVID-19 impact on air quality
2,A global transition to flash droughts under cl...,CN,Science,2023-04-13,Hydrology and Drought Analysis
3,"Climate Change, Firm Performance, and Investor...",US,Management Science,2023-03-21,Market Dynamics and Volatility
4,The global costs of extreme weather that are a...,NZ,Nature Communications,2023-09-29,Climate Change Policy and Economics
5,RETRACTED ARTICLE: The economic commitment of ...,DE,Nature,2024-04-17,Climate Change Policy and Economics
6,Complex plant responses to drought and heat st...,JP,The Plant Journal,2024-01-03,Plant Stress Responses and Tolerance
7,Global concurrent climate extremes exacerbated...,CN,Science Advances,2023-03-10,Climate variability and models
8,National contributions to climate change due t...,GB,Scientific Data,2023-03-29,Atmospheric and Environmental Gas Dynamics
9,Climate Change and Adaptation in Global Supply...,US,Review of Financial Studies,2023-12-09,Supply Chain Resilience and Risk Management


## 6. Save your corpus to disk

`save_corpus` writes two files:
- **`papers.parquet`** — preferred; preserves list columns (like `authors`) and long text cleanly, and is fast to reload.
- **`papers.csv`** — convenient for spreadsheets (lists get stringified).

Reload later with `pd.read_parquet('papers.parquet')`.

In [8]:
save_corpus(df, 'papers')          # -> papers.parquet + papers.csv

# sanity check: reload it
reloaded = pd.read_parquet('papers.parquet')
print('reloaded', len(reloaded), 'rows with columns:', list(reloaded.columns))

2026-06-03 21:43:09,113 INFO Saved papers.parquet + papers.csv (25 rows)


reloaded 25 rows with columns: ['authors', 'title', 'content', 'date', 'country', 'journal', 'theme', 'doi', 'openalex_id', 'abstract', 'pdf_url', 'countries']


## 7. (Optional) Upgrade the theme with an LLM

The free `theme` comes from OpenAlex's topic list. If you'd rather have a custom tag generated by Claude (e.g. tuned to your own taxonomy), set an API key and pass `with_theme=True`. **This is the only step that costs anything / needs a key** — skip it entirely if you don't want it.

```python
import os
os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'      # from console.anthropic.com
df_llm = build_corpus(10, with_theme=True)          # Claude tags each paper
```

It uses the cheap, fast Haiku model and tags one paper per call. For large runs, batch them with the Anthropic Message Batches API.

## 8. Scaling up & troubleshooting

**Bigger corpora**
- Raise `n` — OpenAlex paginates automatically (tens of thousands is fine).
- Downloads are sequential and dominate the time. To go faster, fetch PDFs in parallel: get the papers with `fetch_metadata(n)`, then run `download_fulltext` across a `concurrent.futures.ThreadPoolExecutor`.
- For *very* large pulls, use the OpenAlex bulk data snapshot instead of the API.

**Troubleshooting**

| symptom | cause / fix |
|---|---|
| `ModuleNotFoundError` | Wrong kernel — pick the `.venv` interpreter (top-right). |
| Many `403`/`Not a PDF` warnings | Publisher blocks bots; those papers have no `content`. Filter to recent papers or accept partial coverage. |
| Cell runs forever | Large `n` with downloads is slow; start small, then scale. |
| `content` is empty for a row | No reachable open-access PDF for that paper — metadata is still complete. |
| Want full text only | `df = df[df['content'].notna()]` to drop rows without text. |

**Try it yourself**
1. Build a 30-paper corpus on a topic you care about (`search=...`).
2. Keep only rows that have full text.
3. Count papers per `country`.
4. Save it under a new name with `save_corpus(df, 'my_topic')`.

That's the whole workflow. 🎉